## Imports

In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from imblearn.over_sampling import SMOTE
import pickle

## Utils

In [32]:
def get_data_root():
  '''
    Ritorna il percorso della cartella contenente i dati in base all'ambiente di esecuzione.
  '''
  try:
      import google.colab
      from google.colab import drive

      try:
          drive.mount("/content/drive", force_remount=True)
          return "/content/drive/MyDrive/ColabContent/Data_analytics"
      except Exception:
          print("Drive non montabile")
          return "/content"

  except ImportError:
      return "../../data"

print(get_data_root())

../../data


## Global variables

In [33]:
DATA_ROOT = get_data_root()
DATASET_PATH = f"{DATA_ROOT}/Dataset2526/train.csv"
SEED = 42

# Preprocessing

In [34]:
df = pd.read_csv(DATASET_PATH)

df, df_test = train_test_split(df, test_size=0.2, random_state=SEED)
df_val, df_test = train_test_split(df_test, test_size=0.5, random_state=SEED)

df.to_csv(f"{DATA_ROOT}/train_raw.csv", index=False)
df_val.to_csv(f"{DATA_ROOT}/val_raw.csv", index=False)
df_test.to_csv(f"{DATA_ROOT}/test_raw.csv", index=False)

### Feature selection

In [35]:
# Drop colonne con troppi nan
threshold = 0.25
missing_percent = df.isnull().mean()
cols_to_drop_nan = missing_percent[missing_percent > threshold].index
df_cleaned = df.drop(columns=cols_to_drop_nan)

print(f"Droppate {len(cols_to_drop_nan)} colonne con nan > {threshold*100}%")

Droppate 56 colonne con nan > 25.0%


In [36]:
# Drop colonne poco significative
cols_to_drop_manual = [
    # Testo
    'loan_title',

    # Post-Emissione
    'recoveries_cash', 'collection_recovery_fee', 'total_received_late_fees',
    'loan_status_current_code',
]

df_cleaned = df_cleaned.drop(columns=[c for c in cols_to_drop_manual if c in df_cleaned.columns])

print(f"Droppate {len(cols_to_drop_manual)} colonne poco significative")

Droppate 5 colonne poco significative


### Gestione valori mancanti

In [37]:
num_cols = df_cleaned.select_dtypes(include=[np.number]).columns
cat_cols = df_cleaned.select_dtypes(include=['object']).columns

# Mediana per feature numeriche
for col in num_cols:
  df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

# Moda per feature categoriche
for col in cat_cols:
  if col != 'grade':
    df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])

### Gestione date

In [38]:
# Map per employment_length
emp_len_map = {
    '< 1 year': 0,
    '1 year': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    '8 years': 8,
    '9 years': 9,
    '10+ years': 10
}
df_cleaned['borrower_profile_employment_length'] = df_cleaned['borrower_profile_employment_length'].map(emp_len_map)

# Trasformazione delle date
df_cleaned['loan_issue_date'] = pd.to_datetime(df_cleaned['loan_issue_date'])
df_cleaned['credit_history_earliest_line'] = pd.to_datetime(df_cleaned['credit_history_earliest_line'])

# Calcolo esperienza finanziaria debitore
df_cleaned['months_credit_history'] = (df_cleaned['loan_issue_date'].dt.year - df_cleaned['credit_history_earliest_line'].dt.year) * 12 + \
                              (df_cleaned['loan_issue_date'].dt.month - df_cleaned['credit_history_earliest_line'].dt.month)

# Estrazione mese/anno delle date
date_cols = ['loan_issue_date', 'credit_history_earliest_line', 'last_payment_date', 'last_credit_pull_date']
for col in date_cols:
    if col in df_cleaned.columns:
        date_dt = pd.to_datetime(df_cleaned[col])
        df_cleaned[col + '_year'] = date_dt.dt.year
        df_cleaned[col + '_month'] = date_dt.dt.month

df_cleaned = df_cleaned.drop(columns=date_cols, errors='ignore')

/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_8115/3245697033.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cleaned['loan_issue_date'] = pd.to_datetime(df_cleaned['loan_issue_date'])
/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_8115/3245697033.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cleaned['credit_history_earliest_line'] = pd.to_datetime(df_cleaned['credit_history_earliest_line'])
/var/folders/93/wjn3tmxx3694z231t8jw9z280000gn/T/ipykernel_8115/3245697033.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_dt = pd.t

### Gestione outlier

In [39]:
df_out = df_cleaned.copy()
num_cols = df_out.select_dtypes(include=[np.number]).columns
fico_cols = [c for c in df_out.columns if 'fico' in c.lower()]
perc_cols = [
  'revolving_utilization', 'bankcard_utilization', 'installment_utilization',
  'overall_utilization', 'bankcard_util_gt_75_ratio', 'tradelines_never_delinquent_ratio',
  'loan_contract_interest_rate'
]

for col in num_cols:
  Q1 = df_out[col].quantile(0.25)
  Q3 = df_out[col].quantile(0.75)
  IQR = Q3 - Q1

  # Applichiamo il clipping solo se l'IQR è maggiore di 0
  if IQR > 0:
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_out[col] = df_out[col].clip(lower=lower_bound, upper=upper_bound)
  # Altrimenti solo valori estremi
  else:
    upper_bound = df_out[col].quantile(0.95)
    df_out[col] = df_out[col].clip(upper=upper_bound)

# Gestione valori FICO >850
for col in fico_cols:
  df_out[col] = df_out[col].clip(upper=850)

# Gestione percentuali >100%
for col in perc_cols:
  if col in df_out.columns:
    # Check se percentuale 0-1 o 0-100
    if df_out[col].max() > 1:
      df_out[col] = df_out[col].clip(upper=100)
    else:
      df_out[col] = df_out[col].clip(upper=1.0)

In [40]:
# Drop colonne con varianza 0
cols_to_drop_const = [col for col in df_out.columns if df_out[col].nunique() <= 1]
df_out = df_out.drop(columns=cols_to_drop_const)

print(f"Droppate {len(cols_to_drop_const)} colonne con varianza 0")

Droppate 7 colonne con varianza 0


### Encoding

In [41]:
# Target
grade_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
df_out['grade'] = df_out['grade'].map(grade_map)

# Colonne categoriche
categorical_cols = df_out.select_dtypes(include=['object']).columns

# Dizionari per salvare gli oggetti
oh_encoders = {}
l_encoders = {}

for col in categorical_cols:
    unique_vals = df_out[col].dropna().unique()

    if len(unique_vals) < 15:
        # One-hot
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')
        ohe.fit(df_out[[col]])
        encoded_data = ohe.transform(df_out[[col]])
        column_names = [f"{col}_{cat}" for cat in ohe.categories_[0][1:]]

        encoded_df = pd.DataFrame(encoded_data, columns=column_names, index=df_out.index)
        df_out = pd.concat([df_out.drop(col, axis=1), encoded_df], axis=1)

        oh_encoders[col] = ohe

    else:
        # LabelEncoder
        le = LabelEncoder()
        df_out[col] = le.fit_transform(df_out[col].astype(str))

        l_encoders[col] = le

## Salvataggio asset

In [42]:
final_columns = df_out.drop(columns=['grade']).columns.tolist()

with open('preprocessing_assets.pkl', 'wb') as f:
    pickle.dump({
        'oh_encoders': oh_encoders,
        'l_encoders': l_encoders,
        'cols_to_drop_nan': cols_to_drop_nan,
        'cols_to_drop_manual': cols_to_drop_manual,
        'cols_to_drop_const': cols_to_drop_const,
        'final_columns': final_columns,
        'emp_len_map': emp_len_map,
        'medians': df_out.median().to_dict(),
        'modes': df_out.mode().to_dict(),
        'grade_map': grade_map
    }, f)

## Bilanciamento

In [43]:
X = df_out.drop(columns=['grade'])
y = df_out['grade']

# SMOTE
smote = SMOTE(random_state=SEED)
X_res, y_res = smote.fit_resample(X, y)
df_balanced = pd.DataFrame(X_res, columns=X.columns)
df_balanced['grade'] = y_res

print("Distribuzione dopo SMOTE:")
print(y_res.value_counts())

Distribuzione dopo SMOTE:
grade
2    30142
3    30142
0    30142
1    30142
6    30142
4    30142
5    30142
Name: count, dtype: int64


## Salvataggio train set preprocessato

In [44]:
df_balanced.to_csv(f"{DATA_ROOT}/train_processed.csv", index=False)